
# 00 — Data Inventory and Integrity

## 1. Project Setup and Paths

**Objective:** Initialize the project environment and define the directories used for data analysis and report generation.

This section will:
- Define the root directory of the New Attack Simulation project.
- Create a dedicated report folder for this notebook.
- Verify that the project directory exists.
- Display the configured paths for verification.

**Note:** All original Data Engineering files will remain unchanged. This notebook will only read source data and save its outputs in its own report directory.


In [4]:

# ============================================================
# 1. PROJECT SETUP AND PATHS
# ============================================================

from pathlib import Path

# Main project directory
PROJECT_ROOT = Path(
    r"C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized"
    r"\FYP_Data_Exfiltration\New_Attack_Simulation"
)

# Current notebook
NOTEBOOK_NAME = "00_Data_Inventory_and_Integrity"

# Verify project directory
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f"Project directory not found: {PROJECT_ROOT}"
    )

# Dedicated report directory
REPORT_DIR = PROJECT_ROOT / "reports" / NOTEBOOK_NAME
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Display configuration
print("=" * 60)
print("PROJECT SETUP")
print("=" * 60)

print(f"Project Root : {PROJECT_ROOT}")
print(f"Notebook     : {NOTEBOOK_NAME}")
print(f"Report Folder: {REPORT_DIR}")

print("\nSTATUS: Project paths initialized successfully.")


PROJECT SETUP
Project Root : C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation
Notebook     : 00_Data_Inventory_and_Integrity
Report Folder: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\00_Data_Inventory_and_Integrity

STATUS: Project paths initialized successfully.



## 2. Data Source Discovery

**Objective:** Locate the original Data Engineering directory that will serve as the primary data source for the new attack simulation.

This step will:
- Search the main FYP directory for Data Engineering folders.
- Identify available source directories.
- Display their paths to verify the correct data source.

**Important:** This step is read-only. No original files will be modified, moved, or deleted.


In [5]:

# ============================================================
# 2. DATA SOURCE DISCOVERY
# ============================================================

FYP_ROOT = PROJECT_ROOT.parent

# Find Data Engineering folders (up to 3 levels)
found_dirs = []
visited = set()
search_queue = [(FYP_ROOT, 0)]

while search_queue:
    current_dir, depth = search_queue.pop(0)

    if current_dir in visited:
        continue

    visited.add(current_dir)

    if current_dir.name.lower() == "data_engineering":
        found_dirs.append(current_dir)

    if depth >= 3:
        continue

    for item in current_dir.iterdir():
        if not item.is_dir() or item.is_symlink():
            continue

        if item.name.lower() in {
            ".git", ".venv", "venv",
            "node_modules", "new_attack_simulation"
        }:
            continue

        search_queue.append((item, depth + 1))

# Display results
print("=" * 60)
print("DATA SOURCE DISCOVERY")
print("=" * 60)

print(f"Search Root: {FYP_ROOT}")
print(f"\nData Engineering directories found: {len(found_dirs)}")

for i, directory in enumerate(found_dirs, 1):
    print(f"\n[{i}] {directory}")

if not found_dirs:
    print("\nWARNING: No Data Engineering directory found.")
    print("The source path needs to be specified manually.")
else:
    print("\nSTATUS: Source directory candidates identified.")


DATA SOURCE DISCOVERY
Search Root: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration

Data Engineering directories found: 2

[1] C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Data_Engineering

[2] C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Data_Engineering\Data_engineering

STATUS: Source directory candidates identified.



### 2.1 Data Engineering Directory Structure

**Objective:** Examine the discovered Data Engineering directories to identify the correct source location.

This step will:
- Inspect the structure of both discovered directories.
- Identify folders containing original datasets.
- Locate the Baseline, User-Day, and Feature data directories.
- Help establish the official source path for subsequent analysis.

**Note:** This inspection is read-only. No source files will be modified.


In [6]:

# ============================================================
# 2.1 DATA ENGINEERING DIRECTORY STRUCTURE
# ============================================================

print("=" * 60)
print("DATA ENGINEERING DIRECTORY STRUCTURE")
print("=" * 60)

for i, directory in enumerate(found_dirs, 1):

    print(f"\n[{i}] {directory}")
    print("-" * 60)

    items = sorted(
        directory.iterdir(),
        key=lambda x: (x.is_file(), x.name.lower())
    )

    for item in items:
        label = "[DIR]" if item.is_dir() else "[FILE]"
        print(f"  {label} {item.name}")

    # Inspect data directory if present
    data_path = directory / "data"

    if data_path.is_dir():
        print("\n  DATA SUBDIRECTORIES:")

        for subdir in sorted(data_path.iterdir()):
            if subdir.is_dir():
                files = list(subdir.iterdir())

                print(
                    f"    {subdir.name}/ "
                    f"({len(files)} direct items)"
                )

            else:
                print(f"    [FILE] {subdir.name}")

print("\nSTATUS: Directory structure inspection completed.")


DATA ENGINEERING DIRECTORY STRUCTURE

[1] C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Data_Engineering
------------------------------------------------------------
  [DIR] .git
  [DIR] Data_engineering
  [FILE] .gitignore
  [FILE] README.md

[2] C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Data_Engineering\Data_engineering
------------------------------------------------------------
  [DIR] data
  [DIR] docs
  [DIR] notebooks
  [DIR] results
  [DIR] src
  [FILE] .gitignore
  [FILE] README.md
  [FILE] requirements.txt

  DATA SUBDIRECTORIES:
    features/ (6 direct items)
    processed/ (7 direct items)
    raw/ (1 direct items)

STATUS: Directory structure inspection completed.



### 2.2 Source Data Inventory

**Objective:** Identify and inventory the datasets available in the Data Engineering pipeline.

This step will:
- Establish the official Data Engineering source directory.
- Verify the existence of the raw, processed, and features directories.
- List available datasets with their names, formats, and sizes.
- Identify candidate files for Baseline, User-Day, and Feature analysis.

**Important:** File discovery does not confirm data integrity or provenance. These will be validated in subsequent sections.

**Data Protection:** All source directories are read-only for this analysis. No original datasets will be modified.


In [7]:

# ============================================================
# 2.2 SOURCE DATA INVENTORY
# ============================================================

# Official Data Engineering directory
DATA_ENGINEERING_ROOT = (
    FYP_ROOT / "Data_Engineering" / "Data_engineering"
)

DATA_ROOT = DATA_ENGINEERING_ROOT / "data"

SOURCE_DIRS = {
    "raw": DATA_ROOT / "raw",
    "processed": DATA_ROOT / "processed",
    "features": DATA_ROOT / "features"
}

# Validate source directories
for name, path in SOURCE_DIRS.items():
    if not path.is_dir():
        raise FileNotFoundError(
            f"Missing source directory: {path}"
        )

print("=" * 70)
print("SOURCE DATA INVENTORY")
print("=" * 70)

print(f"\nData Root: {DATA_ROOT}")

file_inventory = []

for category, directory in SOURCE_DIRS.items():

    print(f"\n{'=' * 70}")
    print(f" {category.upper()} DATA")
    print("=" * 70)

    # Inspect raw directory at the first level only
    # to avoid scanning potentially large raw datasets.
    if category == "raw":
        items = sorted(directory.iterdir())
    else:
        items = sorted(directory.rglob("*"))

    files_found = 0

    for item in items:

        if item.is_dir():
            print(f"[DIR]  {item.relative_to(directory)}")
            continue

        if not item.is_file():
            continue

        size_mb = item.stat().st_size / (1024 ** 2)
        relative_path = item.relative_to(DATA_ROOT)

        file_inventory.append({
            "category": category,
            "filename": item.name,
            "relative_path": str(relative_path),
            "format": item.suffix.lower(),
            "size_mb": round(size_mb, 3)
        })

        print(
            f"[FILE] {relative_path} "
            f"| {size_mb:.2f} MB"
        )

        files_found += 1

    print(f"\nFiles discovered: {files_found}")

print("\n" + "=" * 70)
print(f"Total inventoried files: {len(file_inventory)}")
print("STATUS: Source inventory completed.")


SOURCE DATA INVENTORY

Data Root: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\Data_Engineering\Data_engineering\data

 RAW DATA
[FILE] raw\README.md | 0.00 MB

Files discovered: 1

 PROCESSED DATA
[FILE] processed\email_daily_r62.parquet | 11.87 MB
[FILE] processed\ground_truth_events_r62.parquet | 0.02 MB
[FILE] processed\ground_truth_users_r62.parquet | 0.00 MB
[FILE] processed\http_daily_r62.parquet | 1.60 MB
[FILE] processed\logon_daily_r62.parquet | 9.10 MB
[FILE] processed\usb_daily_r62.parquet | 1.02 MB
[FILE] processed\user_day_r62_v1.parquet | 23.51 MB

Files discovered: 7

 FEATURES DATA
[FILE] features\attack_host_eligibility_r62.parquet | 0.03 MB
[FILE] features\behavior_features_r62_v1.parquet | 9.71 MB
[FILE] features\behavior_features_r62_v1.xlsx | 0.00 MB
[FILE] features\malicious_pre_attack_baseline_r62.parquet | 0.01 MB
[FILE] features\user_baseline_r62_v1.csv | 0.89 MB
[FILE] features\user_baseline_r62_v1.parquet | 0.21 MB

Files disco


### 2.3 Core Dataset Metadata and Schema Inspection

**Objective:** Examine the structure and metadata of the core Data Engineering datasets before performing detailed data quality checks.

This step will:
- Identify the primary datasets required for the new attack simulation.
- Retrieve the number of rows and columns in each dataset.
- Inspect column names and data types.
- Identify potential user IDs, dates, and behavioral features.
- Save the dataset inventory and metadata reports.

**Core Datasets:**
- `user_day_r62_v1.parquet` — Daily user activity.
- `user_baseline_r62_v1.parquet` — Historical behavioral baselines.
- `behavior_features_r62_v1.parquet` — Engineered behavioral features.
- `attack_host_eligibility_r62.parquet` — Previous host eligibility information.

**Important:** This inspection uses Parquet metadata without loading complete datasets into memory. Data integrity and dataset provenance will be validated separately.


In [8]:

# ============================================================
# 2.3 CORE DATASET METADATA AND SCHEMA INSPECTION
# ============================================================

import pandas as pd
import pyarrow.parquet as pq

# Core datasets
CORE_DATASETS = {
    "User-Day": (
        DATA_ROOT / "processed" / "user_day_r62_v1.parquet"
    ),
    "Baseline": (
        DATA_ROOT / "features" / "user_baseline_r62_v1.parquet"
    ),
    "Behavior Features": (
        DATA_ROOT / "features" / "behavior_features_r62_v1.parquet"
    ),
    "Host Eligibility": (
        DATA_ROOT / "features" / "attack_host_eligibility_r62.parquet"
    )
}

metadata_records = []
schema_records = []

print("=" * 70)
print("CORE DATASET METADATA AND SCHEMA INSPECTION")
print("=" * 70)

for dataset_name, file_path in CORE_DATASETS.items():

    if not file_path.is_file():
        raise FileNotFoundError(
            f"Missing dataset: {file_path}"
        )

    parquet_file = pq.ParquetFile(file_path)

    rows = parquet_file.metadata.num_rows
    columns = len(parquet_file.schema_arrow.names)

    metadata_records.append({
        "dataset": dataset_name,
        "filename": file_path.name,
        "rows": rows,
        "columns": columns,
        "size_mb": round(
            file_path.stat().st_size / (1024 ** 2), 3
        )
    })

    print(f"\n{'=' * 70}")
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    print(f"File    : {file_path.name}")
    print(f"Rows    : {rows:,}")
    print(f"Columns : {columns}")

    print("\nCOLUMN SCHEMA:")

    for field in parquet_file.schema_arrow:

        schema_records.append({
            "dataset": dataset_name,
            "column": field.name,
            "data_type": str(field.type),
            "nullable": field.nullable
        })

        print(f"  {field.name:<45} {str(field.type)}")

# Convert results to DataFrames
metadata_df = pd.DataFrame(metadata_records)
schema_df = pd.DataFrame(schema_records)

inventory_df = pd.DataFrame(file_inventory)

# Save reports
inventory_df.to_csv(
    REPORT_DIR / "00_data_inventory.csv",
    index=False
)

metadata_df.to_csv(
    REPORT_DIR / "00_core_dataset_metadata.csv",
    index=False
)

schema_df.to_csv(
    REPORT_DIR / "00_core_dataset_schema.csv",
    index=False
)

print("\n" + "=" * 70)
print("DATASET SUMMARY")
print("=" * 70)

print(metadata_df.to_string(index=False))

print("\nReports saved to:")
print(REPORT_DIR)

print("\nSTATUS: Metadata inspection completed.")


CORE DATASET METADATA AND SCHEMA INSPECTION

DATASET: User-Day
File    : user_day_r62_v1.parquet
Rows    : 1,394,010
Columns : 34

COLUMN SCHEMA:
  user                                          string
  day                                           date32[day]
  day_of_week                                   int32
  is_weekend                                    int32
  email_event_count                             int64
  email_send_count                              double
  email_view_count                              double
  external_email_count                          double
  internal_email_count                          double
  total_sent_email_bytes                        double
  external_email_bytes                          double
  emails_with_attachments                       double
  attachment_count                              double
  unique_external_recipients                    int64
  unique_external_domains                       int64
  file_event_count           


## 3. Temporal Integrity Validation

### 3.1 Dataset Date Coverage

**Objective:** Verify that the source datasets follow the official Data Engineering timeline.

**Reference Periods:**
- Baseline: 2010-01-02 to 2010-12-31
- Future / Candidate Attack Period: 2011-01-01 to 2011-06-01
- Baseline Cutoff: 2011-01-01

**This test will:**
- Identify the earliest and latest dates in each dataset.
- Count records within the Baseline and Future periods.
- Count unique users in each dataset.
- Detect records outside the expected date range.
- Verify that Behavior Features do not contain records before the Baseline cutoff.
- Inspect the historical coverage stored in the Baseline dataset.

**Expected Output:**
- `00_temporal_coverage.csv`

**Important:** This test checks date coverage only. Workday eligibility, missing values, duplicate keys, and cross-dataset consistency will be validated separately.

**Data Protection:** All source files are read-only.



In [9]:

# ============================================================
# 3.1 DATASET DATE COVERAGE
# ============================================================

from datetime import date

BASELINE_START = pd.Timestamp("2010-01-02")
BASELINE_END = pd.Timestamp("2010-12-31")

FUTURE_START = pd.Timestamp("2011-01-01")
FUTURE_END = pd.Timestamp("2011-06-01")

# Source datasets and date columns
TEMPORAL_DATASETS = {
    "User-Day": (
        CORE_DATASETS["User-Day"], "day"
    ),
    "Behavior Features": (
        CORE_DATASETS["Behavior Features"], "day"
    )
}

temporal_results = []

print("=" * 70)
print("TEMPORAL INTEGRITY — DATASET DATE COVERAGE")
print("=" * 70)

for name, (path, date_col) in TEMPORAL_DATASETS.items():

    # Load only required columns
    df = pd.read_parquet(
        path,
        columns=["user", date_col]
    )

    df[date_col] = pd.to_datetime(
        df[date_col], errors="coerce"
    )

    dates = df[date_col]

    baseline_mask = dates.between(
        BASELINE_START, BASELINE_END
    )

    future_mask = dates.between(
        FUTURE_START, FUTURE_END
    )

    missing_dates = dates.isna().sum()

    outside_count = (
        (~baseline_mask & ~future_mask & dates.notna())
        .sum()
    )

    temporal_results.append({
        "dataset": name,
        "total_rows": len(df),
        "unique_users": df["user"].nunique(),
        "earliest_date": dates.min(),
        "latest_date": dates.max(),
        "baseline_rows": int(baseline_mask.sum()),
        "future_rows": int(future_mask.sum()),
        "missing_dates": int(missing_dates),
        "outside_period_rows": int(outside_count)
    })

    print(f"\nDATASET: {name}")
    print("-" * 55)
    print(f"Total Rows     : {len(df):,}")
    print(f"Unique Users   : {df['user'].nunique():,}")
    print(f"Earliest Date  : {dates.min()}")
    print(f"Latest Date    : {dates.max()}")
    print(f"Baseline Rows  : {baseline_mask.sum():,}")
    print(f"Future Rows    : {future_mask.sum():,}")
    print(f"Missing Dates  : {missing_dates:,}")
    print(f"Outside Period : {outside_count:,}")

    del df

# Inspect baseline period boundaries
baseline = pd.read_parquet(
    CORE_DATASETS["Baseline"],
    columns=[
        "user",
        "baseline_period_start",
        "baseline_period_end"
    ]
)

baseline_start = pd.to_datetime(
    baseline["baseline_period_start"]
)

baseline_end = pd.to_datetime(
    baseline["baseline_period_end"]
)

print("\n" + "=" * 70)
print("BASELINE DATASET COVERAGE")
print("=" * 70)

print(f"Users            : {baseline['user'].nunique():,}")
print(f"Earliest Start   : {baseline_start.min()}")
print(f"Latest Start     : {baseline_start.max()}")
print(f"Earliest End     : {baseline_end.min()}")
print(f"Latest End       : {baseline_end.max()}")

invalid_baseline = (
    (baseline_start != BASELINE_START)
    | (baseline_end != BASELINE_END)
    | baseline_start.isna()
    | baseline_end.isna()
).sum()

print(f"Unexpected Periods: {invalid_baseline:,}")

# Save report
temporal_df = pd.DataFrame(temporal_results)

temporal_df.to_csv(
    REPORT_DIR / "00_temporal_coverage.csv",
    index=False
)

print("\n" + "=" * 70)
print("TEMPORAL VALIDATION SUMMARY")
print("=" * 70)

for record in temporal_results:

    valid = (
        record["missing_dates"] == 0
        and record["outside_period_rows"] == 0
    )

    if record["dataset"] == "Behavior Features":
        valid = valid and record["baseline_rows"] == 0

    status = "PASS" if valid else "REVIEW"

    print(f"{record['dataset']}: {status}")

print(
    "Baseline Periods:",
    "PASS" if invalid_baseline == 0 else "REVIEW"
)

print("\nReport saved to:", REPORT_DIR)
print("STATUS: Temporal coverage inspection completed.")


TEMPORAL INTEGRITY — DATASET DATE COVERAGE

DATASET: User-Day
-------------------------------------------------------
Total Rows     : 1,394,010
Unique Users   : 4,000
Earliest Date  : 2010-01-02 00:00:00
Latest Date    : 2011-06-01 00:00:00
Baseline Rows  : 998,448
Future Rows    : 395,562
Missing Dates  : 0
Outside Period : 0

DATASET: Behavior Features
-------------------------------------------------------
Total Rows     : 387,162
Unique Users   : 3,737
Earliest Date  : 2011-01-03 00:00:00
Latest Date    : 2011-06-01 00:00:00
Baseline Rows  : 0
Future Rows    : 387,162
Missing Dates  : 0
Outside Period : 0

BASELINE DATASET COVERAGE
Users            : 3,995
Earliest Start   : 2010-01-02 00:00:00
Latest Start     : 2010-01-02 00:00:00
Earliest End     : 2010-12-31 00:00:00
Latest End       : 2010-12-31 00:00:00
Unexpected Periods: 0

TEMPORAL VALIDATION SUMMARY
User-Day: PASS
Behavior Features: PASS
Baseline Periods: PASS

Report saved to: C:\Users\hp\Documents\FYP_Data_Exfiltration


### 3.2 Workday Definition and Eligibility Validation

**Objective:** Verify the official Data Engineering workday rules against actual user activity.

**Official Workday Definition:**
- The date must be Monday through Friday (`is_weekend = 0`).
- The user must have at least one logon event (`logon_count > 0`).
- Both conditions must be satisfied.

**This test will:**
- Verify that weekend flags match actual calendar weekdays.
- Count eligible workdays in the Baseline and Future periods.
- Identify weekdays with no logon activity.
- Identify weekend activity excluded from eligible workdays.
- Verify that all Behavior Features records satisfy the workday definition.
- Compare the workday distributions between User-Day and Behavior Features.

**Expected Output:**
- `00_workday_validation.csv`

**Important:** Workday eligibility is not the same as attack eligibility. No attack scenario constraints are applied in this test.


In [10]:

# ============================================================
# 3.2 WORKDAY DEFINITION AND ELIGIBILITY VALIDATION
# ============================================================

WORKDAY_DATASETS = {
    "User-Day": CORE_DATASETS["User-Day"],
    "Behavior Features": CORE_DATASETS["Behavior Features"]
}

workday_results = []

print("=" * 70)
print("WORKDAY DEFINITION AND ELIGIBILITY VALIDATION")
print("=" * 70)

for dataset_name, file_path in WORKDAY_DATASETS.items():

    df = pd.read_parquet(
        file_path,
        columns=["user", "day", "is_weekend", "logon_count"]
    )

    df["day"] = pd.to_datetime(df["day"])

    # Actual calendar classification
    calendar_weekend = df["day"].dt.dayofweek >= 5

    # Data Engineering classification
    flag_weekend = df["is_weekend"] == 1
    flag_weekday = df["is_weekend"] == 0
    has_logon = df["logon_count"] > 0

    # Validate weekend flags
    valid_flags = df["is_weekend"].isin([0, 1])

    weekend_mismatch = (
        valid_flags
        & df["day"].notna()
        & (calendar_weekend != flag_weekend)
    )

    # Official eligible workday
    eligible_workday = flag_weekday & has_logon

    print(f"\nDATASET: {dataset_name}")
    print("-" * 55)

    print(f"Total rows             : {len(df):,}")
    print(f"Invalid weekend flags  : {(~valid_flags).sum():,}")
    print(f"Weekend flag mismatches: {weekend_mismatch.sum():,}")
    print(f"Missing logon counts   : {df['logon_count'].isna().sum():,}")

    periods = {
        "Baseline": df["day"].between(
            BASELINE_START, BASELINE_END
        ),
        "Future": df["day"].between(
            FUTURE_START, FUTURE_END
        )
    }

    for period_name, period_mask in periods.items():

        total = int(period_mask.sum())

        if total == 0:
            continue

        eligible = period_mask & eligible_workday

        weekday_no_logon = (
            period_mask & flag_weekday & ~has_logon
        )

        weekend_with_logon = (
            period_mask & flag_weekend & has_logon
        )

        result = {
            "dataset": dataset_name,
            "period": period_name,
            "total_rows": total,
            "unique_users": int(
                df.loc[period_mask, "user"].nunique()
            ),
            "eligible_workday_rows": int(eligible.sum()),
            "eligible_users": int(
                df.loc[eligible, "user"].nunique()
            ),
            "weekday_without_logon": int(
                weekday_no_logon.sum()
            ),
            "weekend_with_logon": int(
                weekend_with_logon.sum()
            ),
            "invalid_weekend_flags": int(
                (period_mask & ~valid_flags).sum()
            ),
            "calendar_flag_mismatches": int(
                (period_mask & weekend_mismatch).sum()
            ),
            "missing_logon_counts": int(
                (period_mask & df["logon_count"].isna()).sum()
            )
        }

        workday_results.append(result)

        print(f"\n  PERIOD: {period_name}")
        print(f"  Total Rows          : {total:,}")
        print(f"  Eligible Workdays   : {result['eligible_workday_rows']:,}")
        print(f"  Eligible Users      : {result['eligible_users']:,}")
        print(f"  Weekday No Logon    : {result['weekday_without_logon']:,}")
        print(f"  Weekend With Logon  : {result['weekend_with_logon']:,}")

    # Validate workday rules for Behavior Features
    if dataset_name == "Behavior Features":

        invalid_feature_rows = (
            ~eligible_workday
        ).sum()

        print("\n  BEHAVIOR FEATURES VALIDATION")
        print(f"  Invalid Workday Rows: {invalid_feature_rows:,}")

        print(
            "  STATUS:",
            "PASS" if invalid_feature_rows == 0 else "REVIEW"
        )

    del df

# Save results
workday_df = pd.DataFrame(workday_results)

workday_df.to_csv(
    REPORT_DIR / "00_workday_validation.csv",
    index=False
)

print("\n" + "=" * 70)
print("WORKDAY VALIDATION SUMMARY")
print("=" * 70)

print(workday_df.to_string(index=False))

print("\nReport saved to:", REPORT_DIR)
print("STATUS: Workday validation completed.")


WORKDAY DEFINITION AND ELIGIBILITY VALIDATION

DATASET: User-Day
-------------------------------------------------------
Total rows             : 1,394,010
Invalid weekend flags  : 0
Weekend flag mismatches: 0
Missing logon counts   : 0

  PERIOD: Baseline
  Total Rows          : 998,448
  Eligible Workdays   : 977,519
  Eligible Users      : 4,000
  Weekday No Logon    : 87
  Weekend With Logon  : 20,261

  PERIOD: Future
  Total Rows          : 395,562
  Eligible Workdays   : 387,437
  Eligible Users      : 3,740
  Weekday No Logon    : 38
  Weekend With Logon  : 7,912

DATASET: Behavior Features
-------------------------------------------------------
Total rows             : 387,162
Invalid weekend flags  : 0
Weekend flag mismatches: 0
Missing logon counts   : 0

  PERIOD: Future
  Total Rows          : 387,162
  Eligible Workdays   : 387,162
  Eligible Users      : 3,737
  Weekday No Logon    : 0
  Weekend With Logon  : 0

  BEHAVIOR FEATURES VALIDATION
  Invalid Workday Rows: 0
  


### 3.3 Future Workday and Feature Key Reconciliation

**Objective:** Investigate the difference between eligible future workdays in User-Day and the records available in Behavior Features.

**This test will:**
- Compare eligible User-Day records with Behavior Features using `(user, day)` keys.
- Detect missing or duplicate user-day records.
- Identify users whose eligible workdays are absent from Behavior Features.
- Determine whether excluded users belong to the benign baseline population.
- Distinguish fully excluded users from partially missing records.

**Expected Outputs:**
- `00_workday_key_reconciliation.csv`
- `00_missing_feature_keys.csv`

**Important:** This test investigates dataset coverage differences without modifying source data or changing eligibility rules.


In [11]:

# ============================================================
# 3.3 FUTURE WORKDAY AND FEATURE KEY RECONCILIATION
# ============================================================

# Load required columns only
user_day = pd.read_parquet(
    CORE_DATASETS["User-Day"],
    columns=["user", "day", "is_weekend", "logon_count"]
)

features = pd.read_parquet(
    CORE_DATASETS["Behavior Features"],
    columns=["user", "day"]
)

baseline_users = set(
    pd.read_parquet(
        CORE_DATASETS["Baseline"],
        columns=["user"]
    )["user"]
)

# Standardize dates
user_day["day"] = pd.to_datetime(user_day["day"])
features["day"] = pd.to_datetime(features["day"])

# Official eligible future workdays
eligible = user_day.loc[
    user_day["day"].between(FUTURE_START, FUTURE_END)
    & (user_day["is_weekend"] == 0)
    & (user_day["logon_count"] > 0),
    ["user", "day"]
].copy()

# Validate keys before comparison
for name, df in [
    ("Eligible User-Day", eligible),
    ("Behavior Features", features)
]:
    duplicates = df.duplicated(["user", "day"]).sum()
    missing_keys = df[["user", "day"]].isna().any(axis=1).sum()

    print(
        f"{name}: {duplicates:,} duplicate keys, "
        f"{missing_keys:,} missing keys"
    )

    if duplicates or missing_keys:
        raise ValueError(
            f"Invalid keys detected in {name}"
        )

# Compare exact user-day keys
comparison = eligible.merge(
    features,
    on=["user", "day"],
    how="outer",
    indicator=True,
    validate="one_to_one"
)

missing = comparison.loc[
    comparison["_merge"] == "left_only",
    ["user", "day"]
].copy()

extra = comparison.loc[
    comparison["_merge"] == "right_only",
    ["user", "day"]
].copy()

# Per-user coverage
eligible_counts = eligible.groupby("user").size()
feature_counts = features.groupby("user").size()

user_coverage = pd.DataFrame({
    "eligible_future_days": eligible_counts,
    "feature_days": feature_counts
}).fillna(0).astype(int)

user_coverage["missing_feature_days"] = (
    user_coverage["eligible_future_days"]
    - user_coverage["feature_days"]
)

user_coverage["in_benign_baseline"] = (
    user_coverage.index.isin(baseline_users)
)

user_coverage["coverage_pct"] = (
    user_coverage["feature_days"]
    / user_coverage["eligible_future_days"].replace(0, float("nan"))
    * 100
).round(2)

affected = user_coverage[
    user_coverage["missing_feature_days"] > 0
].copy()

# Display results
print("\n" + "=" * 70)
print("FUTURE WORKDAY RECONCILIATION")
print("=" * 70)

print(f"Eligible User-Day Rows : {len(eligible):,}")
print(f"Behavior Feature Rows  : {len(features):,}")
print(f"Matched Keys           : {(comparison['_merge'] == 'both').sum():,}")
print(f"Missing Feature Keys   : {len(missing):,}")
print(f"Unexpected Feature Keys: {len(extra):,}")

print(f"\nAffected Users: {len(affected):,}")
print(
    "Missing Rows Outside Benign Baseline:",
    (~missing["user"].isin(baseline_users)).sum()
)

print("\nAFFECTED USER DETAILS:")
print(affected.to_string())

# Save reports
user_coverage.reset_index().to_csv(
    REPORT_DIR / "00_workday_key_reconciliation.csv",
    index=False
)

missing.to_csv(
    REPORT_DIR / "00_missing_feature_keys.csv",
    index=False
)

print("\nReports saved to:", REPORT_DIR)
print("STATUS: Key reconciliation completed.")

del user_day, features, comparison


Eligible User-Day: 0 duplicate keys, 0 missing keys
Behavior Features: 0 duplicate keys, 0 missing keys

FUTURE WORKDAY RECONCILIATION
Eligible User-Day Rows : 387,437
Behavior Feature Rows  : 387,162
Matched Keys           : 387,162
Missing Feature Keys   : 275
Unexpected Feature Keys: 0

Affected Users: 3
Missing Rows Outside Benign Baseline: 275

AFFECTED USER DETAILS:
         eligible_future_days  feature_days  missing_feature_days  in_benign_baseline  coverage_pct
user                                                                                               
CDE1846                   105             0                   105               False           0.0
CMP2946                    65             0                    65               False           0.0
MBG3183                   105             0                   105               False           0.0

Reports saved to: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\


### 3.4 Per-User Workday Count Consistency

**Objective:** Verify that workday counts are consistent across the original Data Engineering datasets.

**This test will:**
- Recalculate Baseline and Future workdays for each benign user directly from User-Day.
- Compare recalculated Baseline counts with `user_baseline_r62_v1.parquet`.
- Compare recalculated Baseline and Future counts with `attack_host_eligibility_r62.parquet`.
- Compare Future workday counts with actual Behavior Features records.
- Identify users with inconsistent or missing counts.
- Preserve users with zero Future workdays instead of excluding them.

**Expected Outputs:**
- `00_per_user_workday_consistency.csv`
- `00_workday_count_mismatches.csv`

**Important:** This test validates data consistency only. It does not apply attack eligibility thresholds or exclude users based on their activity levels.


In [12]:

# ============================================================
# 3.4 PER-USER WORKDAY COUNT CONSISTENCY
# ============================================================

print("=" * 70)
print("PER-USER WORKDAY COUNT CONSISTENCY")
print("=" * 70)

# 1. Load required columns only
daily = pd.read_parquet(
    CORE_DATASETS["User-Day"],
    columns=["user", "day", "is_weekend", "logon_count"]
)

baseline = pd.read_parquet(
    CORE_DATASETS["Baseline"],
    columns=["user", "baseline_workdays"]
)

hosts = pd.read_parquet(
    CORE_DATASETS["Host Eligibility"],
    columns=["user", "baseline_workdays", "future_workdays"]
)

features = pd.read_parquet(
    CORE_DATASETS["Behavior Features"],
    columns=["user", "day"]
)

# 2. Check population uniqueness
for name, df in [("Baseline", baseline), ("Host Eligibility", hosts)]:
    if df["user"].isna().any() or df["user"].duplicated().any():
        raise ValueError(f"Invalid user keys in {name}")

if features[["user", "day"]].isna().any().any():
    raise ValueError("Missing user/day keys in Behavior Features")

if features.duplicated(["user", "day"]).any():
    raise ValueError("Duplicate user/day keys in Behavior Features")

# 3. Recalculate official workdays
daily["day"] = pd.to_datetime(daily["day"])

eligible = daily.loc[
    (daily["is_weekend"] == 0)
    & (daily["logon_count"] > 0)
].copy()

baseline_days = eligible.loc[
    eligible["day"].between(BASELINE_START, BASELINE_END)
].groupby("user").size()

future_days = eligible.loc[
    eligible["day"].between(FUTURE_START, FUTURE_END)
].groupby("user").size()

feature_days = features.groupby("user").size()

# 4. Build comparison for every benign user
audit = baseline.rename(
    columns={"baseline_workdays": "baseline_reported"}
).merge(
    hosts.rename(columns={
        "baseline_workdays": "host_baseline_reported",
        "future_workdays": "host_future_reported"
    }),
    on="user",
    how="outer",
    validate="one_to_one",
    indicator=True
)

audit["population_match"] = audit["_merge"] == "both"
audit = audit.drop(columns="_merge")

audit["baseline_calculated"] = audit["user"].map(baseline_days)
audit["future_calculated"] = audit["user"].map(future_days)
audit["feature_days"] = audit["user"].map(feature_days)

for column in [
    "baseline_calculated",
    "future_calculated",
    "feature_days"
]:
    audit[column] = audit[column].fillna(0).astype(int)

# 5. Validate counts
audit["baseline_match"] = (
    (audit["baseline_calculated"] == audit["baseline_reported"])
    & (audit["baseline_calculated"] == audit["host_baseline_reported"])
)

audit["future_match"] = (
    audit["future_calculated"] == audit["host_future_reported"]
)

audit["features_match"] = (
    audit["future_calculated"] == audit["feature_days"]
)

audit["all_checks_pass"] = (
    audit["population_match"]
    & audit["baseline_match"]
    & audit["future_match"]
    & audit["features_match"]
)

mismatches = audit.loc[~audit["all_checks_pass"]].copy()

# 6. Display summary
print(f"\nUsers checked         : {len(audit):,}")
print(f"Population mismatches : {(~audit['population_match']).sum():,}")
print(f"Baseline mismatches   : {(~audit['baseline_match']).sum():,}")
print(f"Future mismatches     : {(~audit['future_match']).sum():,}")
print(f"Feature mismatches    : {(~audit['features_match']).sum():,}")
print(f"Users with zero Future workdays: {(audit['future_calculated'] == 0).sum():,}")

print("\nTOTAL WORKDAY COUNTS")
print("-" * 55)
print(f"Baseline calculated: {audit['baseline_calculated'].sum():,}")
print(f"Future calculated  : {audit['future_calculated'].sum():,}")
print(f"Feature records    : {audit['feature_days'].sum():,}")

if not mismatches.empty:
    print("\nMISMATCH DETAILS (FIRST 20):")
    print(mismatches.head(20).to_string(index=False))

# 7. Save reports
audit.to_csv(
    REPORT_DIR / "00_per_user_workday_consistency.csv",
    index=False
)

mismatches.to_csv(
    REPORT_DIR / "00_workday_count_mismatches.csv",
    index=False
)

print("\n" + "=" * 70)
print(
    "STATUS:",
    "PASS" if mismatches.empty else "REVIEW REQUIRED"
)
print("Reports saved to:", REPORT_DIR)

del daily, eligible, features


PER-USER WORKDAY COUNT CONSISTENCY

Users checked         : 3,995
Population mismatches : 0
Baseline mismatches   : 0
Future mismatches     : 0
Feature mismatches    : 0
Users with zero Future workdays: 258

TOTAL WORKDAY COUNTS
-------------------------------------------------------
Baseline calculated: 976,434
Future calculated  : 387,162
Feature records    : 387,162

STATUS: PASS
Reports saved to: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\00_Data_Inventory_and_Integrity



## 4. Data Quality Validation

### 4.1 Channel Activity Value Integrity

**Objective:** Validate the numerical integrity of the original daily activity measurements across Email, USB, and HTTP channels.

**This test will:**
- Examine event counts, channel activity counts, and data volumes.
- Identify missing, zero, and positive values separately.
- Detect negative values in activity measurements.
- Detect fractional values in event and operation counts.
- Compare data quality between the Baseline and Future periods.
- Preserve the original data without modifying or replacing missing values.

**Primary Attack-Related Measurements:**
- Email: `external_email_count`, `external_email_bytes`
- USB: `usb_copy_count`
- HTTP: `www_upload_count`

**Expected Output:**
- `00_channel_value_integrity.csv`

**Important:** Missing values are not automatically considered errors. Their meaning must be examined before deciding whether they represent inactivity or a data quality issue.

**Data Protection:** This validation is read-only.


In [13]:

# ============================================================
# 4.1 CHANNEL ACTIVITY VALUE INTEGRITY
# ============================================================

import pandas as pd

# Measurements to inspect
CHANNEL_COLUMNS = {
    "Email": [
        "email_event_count",
        "email_send_count",
        "external_email_count",
        "external_email_bytes"
    ],
    "USB": [
        "file_event_count",
        "file_copy_count",
        "usb_copy_count"
    ],
    "HTTP": [
        "http_event_count",
        "www_upload_count"
    ]
}

COUNT_COLUMNS = {
    "email_event_count",
    "email_send_count",
    "external_email_count",
    "file_event_count",
    "file_copy_count",
    "usb_copy_count",
    "http_event_count",
    "www_upload_count"
}

all_columns = [
    column
    for columns in CHANNEL_COLUMNS.values()
    for column in columns
]

# Load required columns only
daily = pd.read_parquet(
    CORE_DATASETS["User-Day"],
    columns=["day"] + all_columns
)

daily["day"] = pd.to_datetime(daily["day"])

periods = {
    "Baseline": daily["day"].between(
        BASELINE_START, BASELINE_END
    ),
    "Future": daily["day"].between(
        FUTURE_START, FUTURE_END
    )
}

quality_results = []

print("=" * 70)
print("CHANNEL ACTIVITY VALUE INTEGRITY")
print("=" * 70)

for period_name, period_mask in periods.items():

    print(f"\n{'=' * 70}")
    print(f"PERIOD: {period_name}")
    print("=" * 70)

    for channel, columns in CHANNEL_COLUMNS.items():

        print(f"\nCHANNEL: {channel}")
        print("-" * 60)

        for column in columns:

            values = daily.loc[period_mask, column]
            non_missing = values.dropna()

            missing_count = int(values.isna().sum())
            zero_count = int((values == 0).sum())
            positive_count = int((values > 0).sum())
            negative_count = int((values < 0).sum())

            fractional_count = 0

            if column in COUNT_COLUMNS:
                fractional_count = int(
                    ((non_missing % 1) != 0).sum()
                )

            quality_results.append({
                "period": period_name,
                "channel": channel,
                "column": column,
                "total_rows": len(values),
                "missing": missing_count,
                "zeros": zero_count,
                "positive": positive_count,
                "negative": negative_count,
                "fractional_counts": fractional_count,
                "minimum": (
                    non_missing.min()
                    if not non_missing.empty else None
                ),
                "maximum": (
                    non_missing.max()
                    if not non_missing.empty else None
                )
            })

            print(
                f"{column:<28} "
                f"Missing: {missing_count:>8,} | "
                f"Negative: {negative_count:>5,} | "
                f"Fractional: {fractional_count:>5,}"
            )

# Build report
quality_df = pd.DataFrame(quality_results)

quality_df.to_csv(
    REPORT_DIR / "00_channel_value_integrity.csv",
    index=False
)

# Identify numerical integrity violations
violations = quality_df[
    (quality_df["negative"] > 0)
    | (quality_df["fractional_counts"] > 0)
]

print("\n" + "=" * 70)
print("DATA QUALITY SUMMARY")
print("=" * 70)

print(f"Measurements Checked : {len(quality_df)}")
print(f"Negative Values      : {quality_df['negative'].sum():,}")
print(f"Fractional Counts    : {quality_df['fractional_counts'].sum():,}")
print(f"Missing Values       : {quality_df['missing'].sum():,}")

if violations.empty:
    print("\nSTATUS: PASS — No numerical integrity violations.")
else:
    print("\nSTATUS: REVIEW REQUIRED")
    print(violations.to_string(index=False))

print("\nReport saved to:", REPORT_DIR)

del daily


CHANNEL ACTIVITY VALUE INTEGRITY

PERIOD: Baseline

CHANNEL: Email
------------------------------------------------------------
email_event_count            Missing:        0 | Negative:     0 | Fractional:     0
email_send_count             Missing:        0 | Negative:     0 | Fractional:     0
external_email_count         Missing:        0 | Negative:     0 | Fractional:     0
external_email_bytes         Missing:        0 | Negative:     0 | Fractional:     0

CHANNEL: USB
------------------------------------------------------------
file_event_count             Missing:        0 | Negative:     0 | Fractional:     0
file_copy_count              Missing:        0 | Negative:     0 | Fractional:     0
usb_copy_count               Missing:        0 | Negative:     0 | Fractional:     0

CHANNEL: HTTP
------------------------------------------------------------
http_event_count             Missing:        0 | Negative:     0 | Fractional:     0
www_upload_count             Missing:    


### 4.2 Cross-Channel Logical Consistency

**Objective:** Validate logical relationships between daily activity measurements using the original Data Engineering rules.

**This test will:**
- Verify that external email counts do not exceed sent emails.
- Verify that internal and external sent emails sum to total sent emails.
- Verify that USB copies do not exceed total file copies.
- Verify that HTTP visits, downloads, and uploads sum to total HTTP events.
- Verify that unique upload domains do not exceed total unique domains.
- Identify missing values in the columns required for these checks.
- Compare violations across the Baseline and Future periods.

**Expected Outputs:**
- `00_channel_logical_consistency.csv`
- `00_channel_logical_violations.csv`

**Important:** These checks validate relationships between recorded activities. They do not evaluate whether a user's behavior is normal, anomalous, or suitable for attack simulation.

**Data Protection:** All source data remains unchanged.


In [14]:

# ============================================================
# 4.2 CROSS-CHANNEL LOGICAL CONSISTENCY
# ============================================================

# Required columns
logical_columns = [
    "user", "day",
    "email_send_count",
    "external_email_count",
    "internal_email_count",
    "file_copy_count",
    "usb_copy_count",
    "http_event_count",
    "www_visit_count",
    "www_download_count",
    "www_upload_count",
    "unique_domains",
    "unique_upload_domains"
]

daily = pd.read_parquet(
    CORE_DATASETS["User-Day"],
    columns=logical_columns
)

daily["day"] = pd.to_datetime(daily["day"])

# Rules derived from the original Data Engineering checks
rules = {
    "External Email <= Sent Email": (
        ["external_email_count", "email_send_count"],
        lambda d: d["external_email_count"] > d["email_send_count"]
    ),
    "Email Partition": (
        ["internal_email_count", "external_email_count", "email_send_count"],
        lambda d: (
            d["internal_email_count"] + d["external_email_count"]
            != d["email_send_count"]
        )
    ),
    "USB Copy <= File Copy": (
        ["usb_copy_count", "file_copy_count"],
        lambda d: d["usb_copy_count"] > d["file_copy_count"]
    ),
    "HTTP Event Partition": (
        [
            "www_visit_count", "www_download_count",
            "www_upload_count", "http_event_count"
        ],
        lambda d: (
            d["www_visit_count"]
            + d["www_download_count"]
            + d["www_upload_count"]
            != d["http_event_count"]
        )
    ),
    "Upload Domains <= Total Domains": (
        ["unique_upload_domains", "unique_domains"],
        lambda d: (
            d["unique_upload_domains"] > d["unique_domains"]
        )
    )
}

periods = {
    "Baseline": daily["day"].between(
        BASELINE_START, BASELINE_END
    ),
    "Future": daily["day"].between(
        FUTURE_START, FUTURE_END
    )
}

results = []
violation_samples = []

print("=" * 70)
print("CROSS-CHANNEL LOGICAL CONSISTENCY")
print("=" * 70)

for period_name, period_mask in periods.items():

    period_df = daily.loc[period_mask]

    print(f"\nPERIOD: {period_name}")
    print("-" * 65)

    for rule_name, (required_cols, check) in rules.items():

        missing_mask = period_df[required_cols].isna().any(axis=1)

        # Evaluate only rows with complete inputs
        violation_mask = (
            check(period_df) & ~missing_mask
        )

        violations = int(violation_mask.sum())
        missing_inputs = int(missing_mask.sum())

        status = (
            "PASS"
            if violations == 0 and missing_inputs == 0
            else "REVIEW"
        )

        results.append({
            "period": period_name,
            "rule": rule_name,
            "rows_checked": len(period_df),
            "violations": violations,
            "missing_inputs": missing_inputs,
            "status": status
        })

        print(
            f"{rule_name:<38} "
            f"Violations: {violations:>6,} | "
            f"Missing: {missing_inputs:>6,} | {status}"
        )

        # Save up to 20 examples per issue type and rule
        for issue, mask in [
            ("RULE_VIOLATION", violation_mask),
            ("MISSING_INPUT", missing_mask)
        ]:
            if mask.any():
                sample = period_df.loc[
                    mask, ["user", "day"] + required_cols
                ].head(20).copy()

                sample["period"] = period_name
                sample["rule"] = rule_name
                sample["issue"] = issue

                violation_samples.append(sample)

# Build reports
logical_df = pd.DataFrame(results)

if violation_samples:
    violations_df = pd.concat(
        violation_samples, ignore_index=True
    )
else:
    violations_df = pd.DataFrame(
        columns=["user", "day", "period", "rule", "issue"]
    )

logical_df.to_csv(
    REPORT_DIR / "00_channel_logical_consistency.csv",
    index=False
)

violations_df.to_csv(
    REPORT_DIR / "00_channel_logical_violations.csv",
    index=False
)

print("\n" + "=" * 70)
print("LOGICAL CONSISTENCY SUMMARY")
print("=" * 70)

print(f"Total Checks   : {len(logical_df)}")
print(f"Violations     : {logical_df['violations'].sum():,}")
print(f"Missing Inputs : {logical_df['missing_inputs'].sum():,}")

overall_pass = (logical_df["status"] == "PASS").all()

print(
    "\nSTATUS:",
    "PASS" if overall_pass else "REVIEW REQUIRED"
)

print("Reports saved to:", REPORT_DIR)

del daily


CROSS-CHANNEL LOGICAL CONSISTENCY

PERIOD: Baseline
-----------------------------------------------------------------
External Email <= Sent Email           Violations:      0 | Missing:      0 | PASS
Email Partition                        Violations:      0 | Missing:      0 | PASS
USB Copy <= File Copy                  Violations:      0 | Missing:      0 | PASS
HTTP Event Partition                   Violations:      0 | Missing:      0 | PASS
Upload Domains <= Total Domains        Violations:      0 | Missing:      0 | PASS

PERIOD: Future
-----------------------------------------------------------------
External Email <= Sent Email           Violations:      0 | Missing:      0 | PASS
Email Partition                        Violations:      0 | Missing:      0 | PASS
USB Copy <= File Copy                  Violations:      0 | Missing:      0 | PASS
HTTP Event Partition                   Violations:      0 | Missing:      0 | PASS
Upload Domains <= Total Domains        Violations:   


### 4.3 Complete Missing and Infinite Value Audit

**Objective:** Examine missing and infinite values across all columns of the core Data Engineering datasets.

**This test will:**
- Inspect every column in User-Day, Baseline, Behavior Features, and Host Eligibility.
- Count missing values and calculate their percentages.
- Detect positive and negative infinity in numeric columns.
- Identify missing user IDs or dates.
- Highlight columns that require further investigation.
- Process large datasets in batches to reduce memory consumption.

**Expected Outputs:**
- `00_missing_infinite_profile.csv`
- `00_missing_infinite_summary.csv`

**Important:** Missing values are not automatically errors. Some baseline statistics may be undefined for users who did not use a specific channel during the baseline period.

This test will identify these cases without replacing or modifying any values.


In [15]:

# ============================================================
# 4.3 COMPLETE MISSING AND INFINITE VALUE AUDIT
# ============================================================

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

profile_records = []
summary_records = []

print("=" * 70)
print("COMPLETE MISSING AND INFINITE VALUE AUDIT")
print("=" * 70)

for dataset_name, file_path in CORE_DATASETS.items():

    parquet_file = pq.ParquetFile(file_path)
    columns = parquet_file.schema_arrow.names

    missing_counts = {col: 0 for col in columns}
    infinite_counts = {col: 0 for col in columns}

    total_rows = 0

    # Read in batches
    for batch in parquet_file.iter_batches(batch_size=50_000):

        chunk = batch.to_pandas()
        total_rows += len(chunk)

        # Missing values
        missing = chunk.isna().sum()

        for col in columns:
            missing_counts[col] += int(missing[col])

        # Infinite numeric values
        numeric = chunk.select_dtypes(include="number")

        if not numeric.empty:
            inf_counts = np.isinf(
                numeric.to_numpy(
                    dtype="float64",
                    na_value=np.nan
                )
            ).sum(axis=0)

            for col, count in zip(numeric.columns, inf_counts):
                infinite_counts[col] += int(count)

    # Generate per-column results
    for col in columns:

        profile_records.append({
            "dataset": dataset_name,
            "column": col,
            "total_rows": total_rows,
            "missing_count": missing_counts[col],
            "missing_pct": round(
                100 * missing_counts[col] / total_rows, 4
            ) if total_rows else 0,
            "infinite_count": infinite_counts[col]
        })

    total_missing = sum(missing_counts.values())
    total_infinite = sum(infinite_counts.values())

    columns_with_missing = sum(
        count > 0 for count in missing_counts.values()
    )

    # Required identifiers
    required_keys = (
        ["user", "day"]
        if dataset_name in ["User-Day", "Behavior Features"]
        else ["user"]
    )

    missing_keys = sum(
        missing_counts.get(col, 0)
        for col in required_keys
    )

    if missing_keys > 0 or total_infinite > 0:
        status = "REVIEW_REQUIRED"
    elif total_missing > 0:
        status = "REVIEW_NULL_SEMANTICS"
    else:
        status = "PASS"

    summary_records.append({
        "dataset": dataset_name,
        "rows": total_rows,
        "columns": len(columns),
        "columns_with_missing": columns_with_missing,
        "total_missing": total_missing,
        "total_infinite": total_infinite,
        "missing_keys": missing_keys,
        "status": status
    })

    print(f"\nDATASET: {dataset_name}")
    print("-" * 60)
    print(f"Rows                 : {total_rows:,}")
    print(f"Columns              : {len(columns)}")
    print(f"Columns with Missing : {columns_with_missing}")
    print(f"Total Missing Values : {total_missing:,}")
    print(f"Infinite Values      : {total_infinite:,}")
    print(f"Missing Keys         : {missing_keys:,}")
    print(f"STATUS               : {status}")

# Save results
profile_df = pd.DataFrame(profile_records)
summary_df = pd.DataFrame(summary_records)

profile_df.to_csv(
    REPORT_DIR / "00_missing_infinite_profile.csv",
    index=False
)

summary_df.to_csv(
    REPORT_DIR / "00_missing_infinite_summary.csv",
    index=False
)

# Show columns that need investigation
affected = profile_df[
    (profile_df["missing_count"] > 0)
    | (profile_df["infinite_count"] > 0)
].copy()

print("\n" + "=" * 70)
print("COLUMNS REQUIRING INVESTIGATION")
print("=" * 70)

if affected.empty:
    print("No missing or infinite values detected.")
else:
    for dataset_name, group in affected.groupby("dataset"):
        print(f"\n{dataset_name}:")
        print(
            group[
                ["column", "missing_count",
                 "missing_pct", "infinite_count"]
            ].sort_values(
                "missing_count", ascending=False
            ).to_string(index=False)
        )

print("\n" + "=" * 70)
print("STATUS: Audit completed.")
print("Reports saved to:", REPORT_DIR)


COMPLETE MISSING AND INFINITE VALUE AUDIT

DATASET: User-Day
------------------------------------------------------------
Rows                 : 1,394,010
Columns              : 34
Columns with Missing : 1
Total Missing Values : 881
Infinite Values      : 0
Missing Keys         : 0
STATUS               : REVIEW_NULL_SEMANTICS

DATASET: Baseline
------------------------------------------------------------
Rows                 : 3,995
Columns              : 36
Columns with Missing : 24
Total Missing Values : 39,564
Infinite Values      : 0
Missing Keys         : 0
STATUS               : REVIEW_NULL_SEMANTICS

DATASET: Behavior Features
------------------------------------------------------------
Rows                 : 387,162
Columns              : 85
Columns with Missing : 10
Total Missing Values : 3,202,340
Infinite Values      : 0
Missing Keys         : 0
STATUS               : REVIEW_NULL_SEMANTICS

DATASET: Host Eligibility
-----------------------------------------------------------


### 4.4 Missing Value Semantics Validation

**Objective:** Determine whether missing values are consistent with the original user activity and baseline definitions.

**This test will:**
- Verify whether missing channel baseline statistics correspond to zero historical active days.
- Check whether missing baseline statistics are consistently propagated into Behavior Features.
- Validate the missing P95 ratio features for USB and HTTP.
- Investigate missing `first_logon` timestamps against daily logon counts.
- Identify unexpected missing or populated values without modifying the original datasets.

**Expected Outputs:**
- `00_null_semantics_checks.csv`
- `00_first_logon_null_review.csv`

**Important:** A missing baseline statistic must not be automatically replaced with zero. A user with no historical activity in a channel may still have activity in the Future period.

**Decision Rule:** Missing values will be classified as expected only when their relationships with the corresponding source measurements are verified.


In [16]:

# ============================================================
# 4.4 MISSING VALUE SEMANTICS VALIDATION
# ============================================================

import pandas as pd

results = []

# ------------------------------------------------------------
# 1. BASELINE NULL SEMANTICS
# ------------------------------------------------------------

baseline = pd.read_parquet(CORE_DATASETS["Baseline"])

BASELINE_GROUPS = {
    "Email": (
        "email_active_days",
        [c for c in baseline.columns
         if c.startswith(("email_count_", "email_bytes_"))]
    ),
    "USB": (
        "usb_active_days",
        [c for c in baseline.columns
         if c.startswith("usb_copy_") and c != "usb_copy_active_days"]
    ),
    "HTTP": (
        "http_upload_active_days",
        [c for c in baseline.columns
         if c.startswith("http_upload_")
         and c != "http_upload_active_days"
         and c != "http_upload_active_rate"]
    )
}

print("=" * 70)
print("BASELINE NULL SEMANTICS")
print("=" * 70)

for channel, (active_col, columns) in BASELINE_GROUPS.items():

    inactive = baseline[active_col].eq(0)

    print(f"\n{channel}")
    print(f"Users with zero active days: {inactive.sum():,}")

    for col in columns:

        # Expected: no historical activity -> undefined statistic
        mismatch = baseline[col].isna() != inactive

        count = int(mismatch.sum())

        results.append({
            "dataset": "Baseline",
            "channel": channel,
            "column": col,
            "mismatches": count,
            "status": "PASS" if count == 0 else "REVIEW"
        })

        print(f"  {col:<33} Mismatches: {count:,}")

# ------------------------------------------------------------
# 2. BEHAVIOR FEATURE NULL PROPAGATION
# ------------------------------------------------------------

FEATURE_GROUPS = {
    "USB": (
        "usb_active_days",
        [
            "usb_copy_p50",
            "usb_copy_p90",
            "usb_copy_p95",
            "usb_copy_mad",
            "usb_copy_to_p95_ratio"
        ]
    ),
    "HTTP": (
        "http_upload_active_days",
        [
            "http_upload_p50",
            "http_upload_p90",
            "http_upload_p95",
            "http_upload_mad",
            "http_upload_to_p95_ratio"
        ]
    )
}

feature_cols = ["user"] + [
    col
    for _, columns in FEATURE_GROUPS.values()
    for col in columns
]

features = pd.read_parquet(
    CORE_DATASETS["Behavior Features"],
    columns=feature_cols
)

baseline_index = baseline.set_index("user")

print("\n" + "=" * 70)
print("BEHAVIOR FEATURE NULL PROPAGATION")
print("=" * 70)

for channel, (active_col, columns) in FEATURE_GROUPS.items():

    expected_null = features["user"].map(
        baseline_index[active_col].eq(0)
    )

    unknown_users = int(expected_null.isna().sum())

    print(f"\n{channel}")
    print(f"Users without baseline mapping (rows): {unknown_users:,}")

    for col in columns:

        mismatch = (
            expected_null.notna()
            & (features[col].isna() != expected_null)
        )

        count = int(mismatch.sum())

        results.append({
            "dataset": "Behavior Features",
            "channel": channel,
            "column": col,
            "mismatches": count + unknown_users,
            "status": (
                "PASS"
                if count == 0 and unknown_users == 0
                else "REVIEW"
            )
        })

        print(f"  {col:<33} Mismatches: {count:,}")

# ------------------------------------------------------------
# 3. FIRST LOGON NULL INVESTIGATION
# ------------------------------------------------------------

logon = pd.read_parquet(
    CORE_DATASETS["User-Day"],
    columns=["user", "day", "logon_count", "first_logon"]
)

missing_first = logon["first_logon"].isna()
has_logon = logon["logon_count"] > 0

missing_with_logon = int(
    (missing_first & has_logon).sum()
)

missing_without_logon = int(
    (missing_first & ~has_logon).sum()
)

logon_review = pd.DataFrame([
    {
        "check": "Missing first_logon with positive logon_count",
        "rows": missing_with_logon
    },
    {
        "check": "Missing first_logon without positive logon_count",
        "rows": missing_without_logon
    },
    {
        "check": "Total missing first_logon",
        "rows": int(missing_first.sum())
    }
])

print("\n" + "=" * 70)
print("FIRST LOGON NULL INVESTIGATION")
print("=" * 70)

print(f"Total Missing First Logon : {missing_first.sum():,}")
print(f"Missing With Logon        : {missing_with_logon:,}")
print(f"Missing Without Logon     : {missing_without_logon:,}")

# ------------------------------------------------------------
# 4. SAVE REPORTS
# ------------------------------------------------------------

results_df = pd.DataFrame(results)

results_df.to_csv(
    REPORT_DIR / "00_null_semantics_checks.csv",
    index=False
)

logon_review.to_csv(
    REPORT_DIR / "00_first_logon_null_review.csv",
    index=False
)

print("\n" + "=" * 70)
print("NULL SEMANTICS SUMMARY")
print("=" * 70)

print(f"Checks Performed : {len(results_df)}")
print(f"Checks Passed    : {(results_df['status'] == 'PASS').sum()}")
print(f"Checks To Review : {(results_df['status'] == 'REVIEW').sum()}")

if missing_with_logon > 0:
    print("First Logon: REVIEW REQUIRED")
else:
    print("First Logon: No missing timestamps with positive logons")

print("\nReports saved to:", REPORT_DIR)

del baseline, features, logon


BASELINE NULL SEMANTICS

Email
Users with zero active days: 1
  email_count_mean_active           Mismatches: 0
  email_count_p50                   Mismatches: 0
  email_count_p75                   Mismatches: 0
  email_count_p90                   Mismatches: 0
  email_count_p95                   Mismatches: 0
  email_count_mad                   Mismatches: 0
  email_bytes_mean_active           Mismatches: 0
  email_bytes_p50                   Mismatches: 0
  email_bytes_p75                   Mismatches: 0
  email_bytes_p90                   Mismatches: 0
  email_bytes_p95                   Mismatches: 0
  email_bytes_mad                   Mismatches: 0

USB
Users with zero active days: 3,212
  usb_copy_mean_active              Mismatches: 0
  usb_copy_p50                      Mismatches: 0
  usb_copy_p75                      Mismatches: 0
  usb_copy_p90                      Mismatches: 0
  usb_copy_p95                      Mismatches: 0
  usb_copy_mad                      Mismatches: 


### 4.5 Primary Key and Population Integrity

**Objective:** Verify the uniqueness and consistency of user identities across the core Data Engineering datasets.

**This test will:**
- Check duplicate and missing primary keys in all four datasets.
- Validate `(user, day)` uniqueness in daily datasets.
- Validate `user` uniqueness in Baseline and Host Eligibility.
- Verify that Baseline and Host Eligibility contain identical user populations.
- Confirm that Behavior Features users belong to the benign Baseline population.
- Identify users without Future feature records.

**Expected Outputs:**
- `00_primary_key_integrity.csv`
- `00_population_relationships.csv`

**Important:** Users without Future records will be documented, not removed. Attack eligibility will be evaluated separately in the next notebook.

**Data Protection:** All original files remain unchanged.


In [17]:

# ============================================================
# 4.5 PRIMARY KEY AND POPULATION INTEGRITY
# ============================================================

KEY_DEFINITIONS = {
    "User-Day": ["user", "day"],
    "Baseline": ["user"],
    "Behavior Features": ["user", "day"],
    "Host Eligibility": ["user"]
}

key_results = []
user_populations = {}

print("=" * 70)
print("PRIMARY KEY AND POPULATION INTEGRITY")
print("=" * 70)

# ------------------------------------------------------------
# 1. PRIMARY KEY VALIDATION
# ------------------------------------------------------------

for dataset_name, key_columns in KEY_DEFINITIONS.items():

    df = pd.read_parquet(
        CORE_DATASETS[dataset_name],
        columns=key_columns
    )

    missing_keys = int(
        df[key_columns].isna().any(axis=1).sum()
    )

    duplicate_keys = int(
        df.duplicated(subset=key_columns).sum()
    )

    unique_users = int(df["user"].nunique())

    user_populations[dataset_name] = set(
        df["user"].dropna().unique()
    )

    status = (
        "PASS"
        if missing_keys == 0 and duplicate_keys == 0
        else "REVIEW"
    )

    key_results.append({
        "dataset": dataset_name,
        "rows": len(df),
        "unique_users": unique_users,
        "primary_key": ", ".join(key_columns),
        "missing_keys": missing_keys,
        "duplicate_keys": duplicate_keys,
        "status": status
    })

    print(f"\nDATASET: {dataset_name}")
    print("-" * 55)
    print(f"Rows           : {len(df):,}")
    print(f"Unique Users   : {unique_users:,}")
    print(f"Missing Keys   : {missing_keys:,}")
    print(f"Duplicate Keys : {duplicate_keys:,}")
    print(f"STATUS         : {status}")

    del df

# ------------------------------------------------------------
# 2. POPULATION RELATIONSHIPS
# ------------------------------------------------------------

daily_users = user_populations["User-Day"]
baseline_users = user_populations["Baseline"]
feature_users = user_populations["Behavior Features"]
host_users = user_populations["Host Eligibility"]

relationships = [
    {
        "check": "Baseline vs Host population difference",
        "count": len(baseline_users ^ host_users),
        "expected": 0
    },
    {
        "check": "Baseline users absent from User-Day",
        "count": len(baseline_users - daily_users),
        "expected": 0
    },
    {
        "check": "Feature users absent from Baseline",
        "count": len(feature_users - baseline_users),
        "expected": 0
    },
    {
        "check": "Feature users absent from User-Day",
        "count": len(feature_users - daily_users),
        "expected": 0
    }
]

population_df = pd.DataFrame(relationships)

population_df["status"] = population_df.apply(
    lambda row: (
        "PASS"
        if row["count"] == row["expected"]
        else "REVIEW"
    ),
    axis=1
)

print("\n" + "=" * 70)
print("POPULATION RELATIONSHIPS")
print("=" * 70)

print(population_df.to_string(index=False))

print("\nADDITIONAL POPULATION INFORMATION")
print("-" * 55)

print(
    "Users outside Benign Baseline:",
    len(daily_users - baseline_users)
)

print(
    "Baseline users without Feature records:",
    len(baseline_users - feature_users)
)

# ------------------------------------------------------------
# 3. SAVE REPORTS
# ------------------------------------------------------------

key_df = pd.DataFrame(key_results)

key_df.to_csv(
    REPORT_DIR / "00_primary_key_integrity.csv",
    index=False
)

population_df.to_csv(
    REPORT_DIR / "00_population_relationships.csv",
    index=False
)

overall_pass = (
    (key_df["status"] == "PASS").all()
    and (population_df["status"] == "PASS").all()
)

print("\n" + "=" * 70)
print("FINAL STATUS:", "PASS" if overall_pass else "REVIEW")
print("Reports saved to:", REPORT_DIR)


PRIMARY KEY AND POPULATION INTEGRITY

DATASET: User-Day
-------------------------------------------------------
Rows           : 1,394,010
Unique Users   : 4,000
Missing Keys   : 0
Duplicate Keys : 0
STATUS         : PASS

DATASET: Baseline
-------------------------------------------------------
Rows           : 3,995
Unique Users   : 3,995
Missing Keys   : 0
Duplicate Keys : 0
STATUS         : PASS

DATASET: Behavior Features
-------------------------------------------------------
Rows           : 387,162
Unique Users   : 3,737
Missing Keys   : 0
Duplicate Keys : 0
STATUS         : PASS

DATASET: Host Eligibility
-------------------------------------------------------
Rows           : 3,995
Unique Users   : 3,995
Missing Keys   : 0
Duplicate Keys : 0
STATUS         : PASS

POPULATION RELATIONSHIPS
                                 check  count  expected status
Baseline vs Host population difference      0         0   PASS
   Baseline users absent from User-Day      0         0   PASS
 


### 4.6 Processed Channel Source Schema Inspection

**Objective:** Inspect the original processed channel datasets before validating their consistency with the integrated User-Day dataset.

**Datasets:**
- `email_daily_r62.parquet`
- `usb_daily_r62.parquet`
- `http_daily_r62.parquet`
- `logon_daily_r62.parquet`

**This test will:**
- Confirm that all four source files exist.
- Retrieve row and column counts.
- Inspect column names and data types.
- Identify user and date keys.
- Determine which activity measurements can be compared with User-Day.

**Expected Outputs:**
- `00_channel_source_metadata.csv`
- `00_channel_source_schema.csv`

**Important:** This is a metadata-only inspection. Actual value reconciliation will be performed in the next step after confirming the source schemas.

**Data Protection:** No original files will be modified.


In [18]:

# ============================================================
# 4.6 PROCESSED CHANNEL SOURCE SCHEMA INSPECTION
# ============================================================

import pyarrow.parquet as pq
import pandas as pd

CHANNEL_SOURCE_FILES = {
    "Email Daily": DATA_ROOT / "processed" / "email_daily_r62.parquet",
    "USB Daily": DATA_ROOT / "processed" / "usb_daily_r62.parquet",
    "HTTP Daily": DATA_ROOT / "processed" / "http_daily_r62.parquet",
    "Logon Daily": DATA_ROOT / "processed" / "logon_daily_r62.parquet"
}

source_metadata = []
source_schema = []

print("=" * 70)
print("PROCESSED CHANNEL SOURCE SCHEMA INSPECTION")
print("=" * 70)

for dataset_name, path in CHANNEL_SOURCE_FILES.items():

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing source dataset: {path}"
        )

    parquet_file = pq.ParquetFile(path)
    schema = parquet_file.schema_arrow

    row_count = parquet_file.metadata.num_rows
    column_count = len(schema)
    size_mb = path.stat().st_size / (1024 ** 2)

    column_names = schema.names

    source_metadata.append({
        "dataset": dataset_name,
        "filename": path.name,
        "rows": row_count,
        "columns": column_count,
        "size_mb": round(size_mb, 3),
        "has_user_key": "user" in column_names,
        "has_day_key": "day" in column_names
    })

    print(f"\n{'=' * 70}")
    print(f"DATASET: {dataset_name}")
    print("=" * 70)

    print(f"File    : {path.name}")
    print(f"Rows    : {row_count:,}")
    print(f"Columns : {column_count}")
    print(f"Size MB : {size_mb:.3f}")

    print("\nCOLUMN SCHEMA:")

    for field in schema:

        source_schema.append({
            "dataset": dataset_name,
            "column": field.name,
            "data_type": str(field.type),
            "nullable": field.nullable
        })

        print(
            f"  {field.name:<40} {str(field.type)}"
        )

# ------------------------------------------------------------
# SAVE REPORTS
# ------------------------------------------------------------

source_metadata_df = pd.DataFrame(source_metadata)
source_schema_df = pd.DataFrame(source_schema)

source_metadata_df.to_csv(
    REPORT_DIR / "00_channel_source_metadata.csv",
    index=False
)

source_schema_df.to_csv(
    REPORT_DIR / "00_channel_source_schema.csv",
    index=False
)

print("\n" + "=" * 70)
print("SOURCE SUMMARY")
print("=" * 70)

print(source_metadata_df.to_string(index=False))

print("\nSTATUS: Source schema inspection completed.")
print("Reports saved to:", REPORT_DIR)


PROCESSED CHANNEL SOURCE SCHEMA INSPECTION

DATASET: Email Daily
File    : email_daily_r62.parquet
Rows    : 1,381,557
Columns : 13
Size MB : 11.872

COLUMN SCHEMA:
  user                                     string
  day                                      date32[day]
  email_event_count                        int64
  email_send_count                         double
  email_view_count                         double
  external_email_count                     double
  internal_email_count                     double
  total_sent_email_bytes                   double
  external_email_bytes                     double
  emails_with_attachments                  double
  attachment_count                         double
  unique_external_recipients               int64
  unique_external_domains                  int64

DATASET: USB Daily
File    : usb_daily_r62.parquet
Rows    : 308,647
Columns : 11
Size MB : 1.016

COLUMN SCHEMA:
  user                                     string
  day             


### 4.7 Processed Source-to-User-Day Reconciliation

**Objective:** Verify that all daily channel measurements were correctly preserved when the original Data Engineering pipeline created the unified User-Day dataset.

**Original Integration Rules:**
- User-Day keys are generated from the union of all four channel datasets.
- Missing numerical channel measurements are replaced with zero during integration.
- Original numerical values must remain unchanged.
- Logon timestamps retain their original values, including nulls.

**This test will:**
- Verify primary key integrity in every processed source.
- Detect source records missing from User-Day.
- Identify User-Day records without corresponding channel activity.
- Compare every source measurement with its integrated User-Day value.
- Identify and document any discrepancies.

**Expected Outputs:**
- `00_source_reconciliation_summary.csv`
- `00_source_column_reconciliation.csv`
- `00_source_reconciliation_mismatches.csv`

**Validation Criteria:**
- No duplicate or missing source keys.
- Every source key exists in User-Day.
- Zero mismatches between actual and expected values.

**Important:** Missing channel records are expected for inactive channels and must not be treated as data errors.

**Data Protection:** All source files remain unchanged.


In [19]:

# ============================================================
# 4.7 PROCESSED SOURCE-TO-USER-DAY RECONCILIATION
# ============================================================

import duckdb
import pandas as pd
import pyarrow.parquet as pq

con = duckdb.connect()

# Safely format local Parquet paths for SQL
def parquet_path(path):
    return str(path).replace("\\", "/").replace("'", "''")

def quote_column(name):
    return '"' + name.replace('"', '""') + '"'

user_day_path = parquet_path(CORE_DATASETS["User-Day"])

summary_records = []
column_records = []
mismatch_samples = []

print("=" * 70)
print("PROCESSED SOURCE-TO-USER-DAY RECONCILIATION")
print("=" * 70)

try:
    for source_name, source_path in CHANNEL_SOURCE_FILES.items():

        source_sql_path = parquet_path(source_path)

        # Inspect source columns dynamically
        source_columns = pq.ParquetFile(
            source_path
        ).schema_arrow.names

        measurement_columns = [
            col for col in source_columns
            if col not in ["user", "day"]
        ]

        # ----------------------------------------------------
        # 1. SOURCE KEY INTEGRITY
        # ----------------------------------------------------

        source_rows, missing_keys, duplicate_keys = con.execute(f"""
            SELECT
                COUNT(*),
                COUNT(*) FILTER (
                    WHERE user IS NULL OR day IS NULL
                ),
                COUNT(*) - COUNT(DISTINCT (user, day))
            FROM read_parquet('{source_sql_path}')
        """).fetchone()

        # Every source key should exist in User-Day
        source_only = con.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{source_sql_path}') s
            ANTI JOIN read_parquet('{user_day_path}') u
                USING (user, day)
        """).fetchone()[0]

        # Missing source rows are allowed for inactive channels
        merged_without_source = con.execute(f"""
            SELECT COUNT(*)
            FROM read_parquet('{user_day_path}') u
            ANTI JOIN read_parquet('{source_sql_path}') s
                USING (user, day)
        """).fetchone()[0]

        # ----------------------------------------------------
        # 2. COMPARE ALL SOURCE MEASUREMENTS
        # ----------------------------------------------------

        total_mismatches = 0

        if missing_keys == 0 and duplicate_keys == 0:

            comparison_expressions = []

            for col in measurement_columns:

                qcol = quote_column(col)

                # Timestamps preserve NULL.
                # Numeric activity measurements use COALESCE(0).
                if col in ["first_logon", "last_logoff"]:
                    expected = f"s.{qcol}"
                else:
                    expected = f"COALESCE(s.{qcol}, 0)"

                comparison_expressions.append(
                    f"""COUNT(*) FILTER (
                        WHERE u.{qcol} IS DISTINCT FROM {expected}
                    ) AS {qcol}"""
                )

            comparison_sql = ",\n".join(
                comparison_expressions
            )

            mismatch_counts = con.execute(f"""
                SELECT {comparison_sql}
                FROM read_parquet('{user_day_path}') u
                LEFT JOIN read_parquet('{source_sql_path}') s
                    USING (user, day)
            """).fetchone()

            for col, mismatch_count in zip(
                measurement_columns, mismatch_counts
            ):

                mismatch_count = int(mismatch_count)
                total_mismatches += mismatch_count

                column_records.append({
                    "source": source_name,
                    "column": col,
                    "mismatches": mismatch_count,
                    "status": (
                        "PASS" if mismatch_count == 0
                        else "REVIEW"
                    )
                })

                if mismatch_count > 0:

                    qcol = quote_column(col)

                    expected = (
                        f"s.{qcol}"
                        if col in ["first_logon", "last_logoff"]
                        else f"COALESCE(s.{qcol}, 0)"
                    )

                    sample = con.execute(f"""
                        SELECT
                            u.user,
                            u.day,
                            u.{qcol} AS merged_value,
                            s.{qcol} AS source_value
                        FROM read_parquet('{user_day_path}') u
                        LEFT JOIN read_parquet('{source_sql_path}') s
                            USING (user, day)
                        WHERE u.{qcol}
                            IS DISTINCT FROM {expected}
                        LIMIT 5
                    """).df()

                    sample["source"] = source_name
                    sample["column"] = col

                    mismatch_samples.append(sample)

        # ----------------------------------------------------
        # 3. SOURCE SUMMARY
        # ----------------------------------------------------

        status = (
            "PASS"
            if (
                missing_keys == 0
                and duplicate_keys == 0
                and source_only == 0
                and total_mismatches == 0
            )
            else "REVIEW"
        )

        summary_records.append({
            "source": source_name,
            "source_rows": source_rows,
            "columns_checked": len(measurement_columns),
            "missing_keys": missing_keys,
            "duplicate_keys": duplicate_keys,
            "source_keys_absent_from_user_day": source_only,
            "user_day_keys_without_source": merged_without_source,
            "value_mismatches": total_mismatches,
            "status": status
        })

        print(f"\nSOURCE: {source_name}")
        print("-" * 60)
        print(f"Source Rows            : {source_rows:,}")
        print(f"Measurements Checked   : {len(measurement_columns)}")
        print(f"Missing Keys           : {missing_keys:,}")
        print(f"Duplicate Keys         : {duplicate_keys:,}")
        print(f"Source Keys Missing    : {source_only:,}")
        print(f"User-Day Without Source: {merged_without_source:,}")
        print(f"Value Mismatches       : {total_mismatches:,}")
        print(f"STATUS                 : {status}")

finally:
    con.close()

# ------------------------------------------------------------
# 4. SAVE REPORTS
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary_records)
column_df = pd.DataFrame(column_records)

if mismatch_samples:
    mismatch_df = pd.concat(
        mismatch_samples, ignore_index=True
    )
else:
    mismatch_df = pd.DataFrame(columns=[
        "user", "day", "merged_value",
        "source_value", "source", "column"
    ])

summary_df.to_csv(
    REPORT_DIR / "00_source_reconciliation_summary.csv",
    index=False
)

column_df.to_csv(
    REPORT_DIR / "00_source_column_reconciliation.csv",
    index=False
)

mismatch_df.to_csv(
    REPORT_DIR / "00_source_reconciliation_mismatches.csv",
    index=False
)

print("\n" + "=" * 70)
print("FINAL SOURCE RECONCILIATION SUMMARY")
print("=" * 70)

print(f"Sources Checked      : {len(summary_df)}")
print(f"Measurements Checked : {len(column_df)}")
print(f"Total Value Mismatches: {summary_df['value_mismatches'].sum():,}")

overall_pass = (summary_df["status"] == "PASS").all()

print("\nFINAL STATUS:", "PASS" if overall_pass else "REVIEW REQUIRED")
print("Reports saved to:", REPORT_DIR)


PROCESSED SOURCE-TO-USER-DAY RECONCILIATION

SOURCE: Email Daily
------------------------------------------------------------
Source Rows            : 1,381,557
Measurements Checked   : 11
Missing Keys           : 0
Duplicate Keys         : 0
Source Keys Missing    : 0
User-Day Without Source: 12,453
Value Mismatches       : 0
STATUS                 : PASS

SOURCE: USB Daily
------------------------------------------------------------
Source Rows            : 308,647
Measurements Checked   : 9
Missing Keys           : 0
Duplicate Keys         : 0
Source Keys Missing    : 0
User-Day Without Source: 1,085,363
Value Mismatches       : 0
STATUS                 : PASS

SOURCE: HTTP Daily
------------------------------------------------------------
Source Rows            : 1,391,239
Measurements Checked   : 6
Missing Keys           : 0
Duplicate Keys         : 0
Source Keys Missing    : 0
User-Day Without Source: 2,771
Value Mismatches       : 0
STATUS                 : PASS

SOURCE: Logon D


## 5. Final Data Quality Assessment

### 5.1 Consolidated QA Report

**Objective:** Consolidate all completed validation results into a reproducible Data Quality Assessment for the New Attack Simulation project.

**This step will:**
- Review previously generated validation reports.
- Consolidate test outcomes into a single QA checklist.
- Document verified dataset sizes and user populations.
- Record important data characteristics affecting future attack simulation.
- Identify remaining limitations and checks not covered by this notebook.
- Generate a final Markdown report.

**Expected Outputs:**
- `00_final_qa_summary.csv`
- `00_data_integrity_summary.md`

**Completion Criteria:** All required integrity checks must pass before the project proceeds to detailed user population analysis.

**Scope Limitation:** Passing these checks establishes the integrity of the inspected processed datasets. It does not independently reproduce all feature-engineering calculations or verify the unavailable raw event files.


In [20]:

# ============================================================
# 5.1 CONSOLIDATED DATA QUALITY ASSESSMENT
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. LOAD PREVIOUS VALIDATION REPORTS
# ------------------------------------------------------------

REPORT_FILES = {
    "temporal": "00_temporal_coverage.csv",
    "workday": "00_workday_validation.csv",
    "counts": "00_per_user_workday_consistency.csv",
    "values": "00_channel_value_integrity.csv",
    "logical": "00_channel_logical_consistency.csv",
    "missing": "00_missing_infinite_summary.csv",
    "nulls": "00_null_semantics_checks.csv",
    "logon": "00_first_logon_null_review.csv",
    "keys": "00_primary_key_integrity.csv",
    "population": "00_population_relationships.csv",
    "sources": "00_source_reconciliation_summary.csv",
    "source_columns": "00_source_column_reconciliation.csv"
}

reports = {}

for name, filename in REPORT_FILES.items():
    path = REPORT_DIR / filename

    if not path.is_file():
        raise FileNotFoundError(
            f"Missing QA report: {path}"
        )

    reports[name] = pd.read_csv(path)

# ------------------------------------------------------------
# 2. BUILD QA CHECKLIST
# ------------------------------------------------------------

qa_records = []

def add_check(name, passed, details):
    qa_records.append({
        "check": name,
        "status": "PASS" if passed else "REVIEW",
        "details": details
    })

temporal = reports["temporal"]
workday = reports["workday"]
counts = reports["counts"]
values = reports["values"]
logical = reports["logical"]
missing = reports["missing"]
nulls = reports["nulls"]
logon = reports["logon"]
keys = reports["keys"]
population = reports["population"]
sources = reports["sources"]
source_columns = reports["source_columns"]

# Temporal integrity
add_check(
    "Temporal Coverage",
    (
        temporal["missing_dates"].sum() == 0
        and temporal["outside_period_rows"].sum() == 0
        and temporal.loc[
            temporal["dataset"] == "Behavior Features",
            "baseline_rows"
        ].sum() == 0
    ),
    "No missing dates, out-of-range rows, or baseline feature rows."
)

# Baseline period boundaries
baseline_periods = pd.read_parquet(
    CORE_DATASETS["Baseline"],
    columns=["baseline_period_start", "baseline_period_end"]
)

valid_baseline_periods = (
    pd.to_datetime(
        baseline_periods["baseline_period_start"]
    ).eq(BASELINE_START).all()
    and pd.to_datetime(
        baseline_periods["baseline_period_end"]
    ).eq(BASELINE_END).all()
)

add_check(
    "Baseline Period Boundaries",
    valid_baseline_periods,
    "All baseline records use the official reference period."
)

# Workday definition
add_check(
    "Workday Calendar Consistency",
    (
        workday["invalid_weekend_flags"].sum() == 0
        and workday["calendar_flag_mismatches"].sum() == 0
        and workday["missing_logon_counts"].sum() == 0
    ),
    "Weekend classifications and logon availability are valid."
)

feature_workdays = workday[
    workday["dataset"] == "Behavior Features"
]

add_check(
    "Behavior Feature Workdays",
    (
        len(feature_workdays) == 1
        and (
            feature_workdays["eligible_workday_rows"]
            == feature_workdays["total_rows"]
        ).all()
    ),
    "All feature records satisfy the official workday rule."
)

# Per-user consistency
add_check(
    "Per-User Workday Counts",
    counts["all_checks_pass"].all(),
    "Baseline, Future, and Feature workday counts agree."
)

# Numerical integrity
add_check(
    "Channel Numerical Integrity",
    (
        values["negative"].sum() == 0
        and values["fractional_counts"].sum() == 0
        and values["missing"].sum() == 0
    ),
    "No invalid values in inspected channel measurements."
)

# Logical integrity
add_check(
    "Channel Logical Consistency",
    (logical["status"] == "PASS").all(),
    "All ten channel relationship checks passed."
)

# Missing and infinite values
add_check(
    "Missing Keys and Infinite Values",
    (
        missing["missing_keys"].sum() == 0
        and missing["total_infinite"].sum() == 0
    ),
    "No missing primary keys or infinite numeric values."
)

# Null semantics
add_check(
    "Baseline and Feature Null Semantics",
    (
        len(nulls) == 34
        and (nulls["status"] == "PASS").all()
    ),
    "All 34 historical activity and null propagation checks passed."
)

logon_issue = logon.loc[
    logon["check"]
    == "Missing first_logon with positive logon_count",
    "rows"
]

add_check(
    "First Logon Null Semantics",
    (
        len(logon_issue) == 1
        and logon_issue.iloc[0] == 0
    ),
    "No missing first_logon timestamps on positive-logon days."
)

# Keys and populations
add_check(
    "Primary Key Integrity",
    (keys["status"] == "PASS").all(),
    "No duplicate or missing primary keys."
)

add_check(
    "Population Relationships",
    (population["status"] == "PASS").all(),
    "Dataset user populations follow expected relationships."
)

# Source reconciliation
add_check(
    "Processed Source Reconciliation",
    (
        len(sources) == 4
        and sources["columns_checked"].sum() == 30
        and (sources["status"] == "PASS").all()
        and (source_columns["status"] == "PASS").all()
    ),
    "All four source datasets match User-Day across 30 measurements."
)

# ------------------------------------------------------------
# 3. FINAL QA RESULTS
# ------------------------------------------------------------

qa_df = pd.DataFrame(qa_records)

passed = int((qa_df["status"] == "PASS").sum())
review = int((qa_df["status"] == "REVIEW").sum())

overall_status = (
    "PASS" if review == 0 else "REVIEW REQUIRED"
)

print("=" * 70)
print("MUBSIR — FINAL DATA INTEGRITY QA")
print("=" * 70)

print(f"\nChecks Completed : {len(qa_df)}")
print(f"Checks Passed    : {passed}")
print(f"Checks To Review : {review}")
print(f"Overall Status   : {overall_status}")

print("\nQA CHECKLIST")
print("-" * 70)

print(
    qa_df[["check", "status"]].to_string(index=False)
)

# ------------------------------------------------------------
# 4. RECORD VERIFIED POPULATION STATISTICS
# ------------------------------------------------------------

benign_users = len(counts)
future_users = int((counts["future_calculated"] > 0).sum())
zero_future = int((counts["future_calculated"] == 0).sum())

baseline_workdays = int(
    counts["baseline_calculated"].sum()
)

future_workdays = int(
    counts["future_calculated"].sum()
)

# ------------------------------------------------------------
# 5. SAVE FINAL CSV
# ------------------------------------------------------------

qa_df.to_csv(
    REPORT_DIR / "00_final_qa_summary.csv",
    index=False
)

# ------------------------------------------------------------
# 6. GENERATE FINAL MARKDOWN DOCUMENT
# ------------------------------------------------------------

summary_md = f"""# MUBSIR — Data Inventory and Integrity Report

## 1. Assessment Overview

- Notebook: `00_Data_Inventory_and_Integrity`
- Dataset Version: CERT r6.2
- QA Status: **{overall_status}**
- Checks Completed: {len(qa_df)}
- Checks Passed: {passed}
- Checks Requiring Review: {review}

## 2. Official Data Periods

- Baseline Start: 2010-01-02
- Baseline End: 2010-12-31
- Future Start: 2011-01-01
- Future End: 2011-06-01

## 3. Population Summary

- Benign Baseline Users: {benign_users:,}
- Users With Future Workdays: {future_users:,}
- Users Without Future Workdays: {zero_future:,}
- Baseline Workday Records: {baseline_workdays:,}
- Future Feature Records: {future_workdays:,}

## 4. Validation Results

| Check | Status |
|---|---|
"""

for _, record in qa_df.iterrows():
    summary_md += (
        f"| {record['check']} | {record['status']} |\n"
    )

summary_md += """
## 5. Findings Relevant to Attack Simulation

- Workdays require a weekday and positive logon activity.
- No attack eligibility threshold has been applied.
- Missing USB and HTTP baseline statistics are associated
  with zero historical activity in the corresponding channel.
- Missing baseline statistics must not be replaced with zero
  without explicit justification.
- Users without Future workdays remain documented.
- The original processed channel measurements match User-Day.
- Off-days in future attack scenarios will mean days without
  injection, not days without natural activity.

## 6. Validation Scope and Limitations

- Original raw event files are unavailable in the inspected
  Data Engineering raw directory.
- The checks validate processed-data integrity and consistency.
- All engineered feature formulas have not been independently
  recomputed.
- The absence of pre-existing malicious activity in every
  candidate user-day has not been independently established.
- Attack feasibility, switching, stealth, and scenario eligibility
  will be evaluated in subsequent notebooks.

## 7. Next Step

Proceed to `01_User_Population_Audit` if all required QA
checks pass.

The next notebook will analyze the available user population
without imposing the previous attack scenario constraints.
"""

summary_path = (
    REPORT_DIR / "00_data_integrity_summary.md"
)

summary_path.write_text(
    summary_md,
    encoding="utf-8"
)

print("\n" + "=" * 70)
print("REPORT GENERATION")
print("=" * 70)

print("Final QA CSV:", REPORT_DIR / "00_final_qa_summary.csv")
print("Final Markdown:", summary_path)

print("\nFINAL STATUS:", overall_status)


MUBSIR — FINAL DATA INTEGRITY QA

Checks Completed : 13
Checks Passed    : 13
Checks To Review : 0
Overall Status   : PASS

QA CHECKLIST
----------------------------------------------------------------------
                              check status
                  Temporal Coverage   PASS
         Baseline Period Boundaries   PASS
       Workday Calendar Consistency   PASS
          Behavior Feature Workdays   PASS
            Per-User Workday Counts   PASS
        Channel Numerical Integrity   PASS
        Channel Logical Consistency   PASS
   Missing Keys and Infinite Values   PASS
Baseline and Feature Null Semantics   PASS
         First Logon Null Semantics   PASS
              Primary Key Integrity   PASS
           Population Relationships   PASS
    Processed Source Reconciliation   PASS

REPORT GENERATION
Final QA CSV: C:\Users\hp\Documents\FYP_Data_Exfiltration_Organized\FYP_Data_Exfiltration\New_Attack_Simulation\reports\00_Data_Inventory_and_Integrity\00_final_qa_summary